In [13]:
import pandas as pd
import numpy as np

def generate_performance_table(csv_file_path, column_name, aggregation='mean', crypto_type='signature'):
    """
    Generate 2D table with crypto type, algorithm, security level, and text sizes.
    
    Args:
        csv_file_path (str): Path to CSV file
        column_name (str): Column to analyze
        aggregation (str): 'mean', 'median', 'min', 'max'
        crypto_type (str): 'signature' or 'kem'
    """
    # Load data
    data = pd.read_csv(csv_file_path)
    
    # Define common algorithms based on crypto type
    if crypto_type.lower() == 'signature':
        crypto_algorithms = {
            'Lattice-based': ['Dilithium2', 'Falcon-512'],
            'Hash-based': ['SPHINCS+-SHA2-128f-simple', 'SPHINCS+-SHAKE-128f-simple'],
            'Multivariate': ['MAYO-1', 'MAYO-2'],
            'Code-based': ['cross-rsdp-128-balanced', 'cross-rsdp-128-fast']
        }
        expected_type = 'Signature'
    elif crypto_type.lower() == 'kem':
        crypto_algorithms = {
            'Lattice-based': ['Kyber512', 'Kyber768', 'Kyber1024'],
            'Code-based': ['Classic-McEliece-348864', 'Classic-McEliece-460896'],
            'Isogeny-based': ['SIKE-p434', 'SIKE-p610'] if 'SIKE' in str(data['algorithm'].unique()) else [],
            'Other': ['FrodoKEM-640-AES', 'NTRU-HPS-2048-509']
        }
        expected_type = 'KEM'
    else:
        return f"Error: crypto_type must be 'signature' or 'kem'"
    
    # Filter out empty algorithm lists
    crypto_algorithms = {k: v for k, v in crypto_algorithms.items() if v}
    
    # Get all common algorithms
    all_common_algorithms = []
    for algorithms in crypto_algorithms.values():
        all_common_algorithms.extend(algorithms)
    
    # Filter data
    data = data[
        (data['algorithm'].isin(all_common_algorithms)) &
        (data['type'] == expected_type) &
        (data['correctness'] == True) & 
        (data['error'].isna() | (data['error'] == ''))
    ].copy()
    
    if len(data) == 0:
        return f"No data found for {crypto_type} algorithms"
    
    if column_name not in data.columns:
        return f"Error: Column '{column_name}' not found. Available columns: {list(data.columns)}"
    
    # Get text sizes
    text_sizes = sorted(data['text_size_kb'].unique())
    
    # Aggregate data
    if aggregation == 'mean':
        agg_data = data.groupby(['algorithm', 'text_size_kb'])[column_name].mean().unstack(fill_value='-')
    elif aggregation == 'median':
        agg_data = data.groupby(['algorithm', 'text_size_kb'])[column_name].median().unstack(fill_value='-')
    elif aggregation == 'min':
        agg_data = data.groupby(['algorithm', 'text_size_kb'])[column_name].min().unstack(fill_value='-')
    elif aggregation == 'max':
        agg_data = data.groupby(['algorithm', 'text_size_kb'])[column_name].max().unstack(fill_value='-')
    
    # Get security levels for each algorithm
    security_info = data.groupby('algorithm')['security_level_bits'].first().to_dict()
    
    # Generate markdown table
    markdown_lines = []
    title = f"{column_name.replace('_', ' ').title()} for {crypto_type.title()} Algorithms ({aggregation.title()})"
    markdown_lines.append(f"# {title}\n")
    
    # Header
    header = "| Cryptography Type | Algorithm | Security Level |"
    for size in text_sizes:
        header += f" {size} KB |"
    markdown_lines.append(header)
    
    # Separator
    separator = "|---|---|---|"
    for _ in text_sizes:
        separator += "---|"
    markdown_lines.append(separator)
    
    # Data rows
    for family, algorithms in crypto_algorithms.items():
        first_in_group = True
        
        for algorithm in algorithms:
            if algorithm in agg_data.index:
                # Shorten algorithm name for better display
                display_name = (algorithm.replace('SPHINCS+-', 'SPX-')
                                        .replace('-simple', '')
                                        .replace('cross-', '')
                                        .replace('Classic-McEliece-', 'CME-')
                                        .replace('FrodoKEM-', 'Frodo-')
                                        .replace('ML-KEM-', 'MLKEM-'))
                
                # Get security level
                sec_level = security_info.get(algorithm, 'N/A')
                sec_level_str = f"{sec_level} bits" if sec_level != 'N/A' else 'N/A'
                
                # Show crypto type only for first algorithm
                if first_in_group:
                    row = f"| **{family}** | {display_name} | {sec_level_str} |"
                    first_in_group = False
                else:
                    row = f"| | {display_name} | {sec_level_str} |"
                
                # Add performance data
                for size in text_sizes:
                    if size in agg_data.columns:
                        value = agg_data.loc[algorithm, size]
                        if value == '-' or pd.isna(value):
                            row += " - |"
                        else:
                            # Format based on data type
                            if 'time' in column_name.lower():
                                row += f" {value:.3f} |"
                            elif 'length' in column_name.lower() or 'bytes' in column_name.lower():
                                row += f" {value:.0f} |"
                            else:
                                row += f" {value:.3f} |"
                    else:
                        row += " - |"
                
                markdown_lines.append(row)
    
    return "\n".join(markdown_lines)

def generate_signature_tables(csv_file_path, aggregation='mean'):
    """Generate all 4 signature performance tables."""
    
    tables = {
        'keygen_time_ms': 'Key Generation Time',
        'sign_time_ms': 'Signing Time',
        'verify_time_ms': 'Verification Time',
        'overhead_bytes': 'Overhead'
    }
    
    all_tables = []
    
    for column_name, table_name in tables.items():
        print(f"Generating {table_name} table for signatures...")
        table = generate_performance_table(csv_file_path, column_name, aggregation, 'signature')
        all_tables.append(table)
        all_tables.append("\n" + "="*80 + "\n")
    
    return "\n".join(all_tables)

def generate_kem_tables(csv_file_path, aggregation='mean'):
    """Generate all 4 KEM performance tables."""
    
    tables = {
        'keygen_time_ms': 'Key Generation Time',
        'encap_time_ms': 'Encapsulation Time',
        'decap_time_ms': 'Decapsulation Time',
        'overhead_bytes': 'Overhead'
    }
    
    all_tables = []
    
    for column_name, table_name in tables.items():
        print(f"Generating {table_name} table for KEM...")
        table = generate_performance_table(csv_file_path, column_name, aggregation, 'kem')
        all_tables.append(table)
        all_tables.append("\n" + "="*80 + "\n")
    
    return "\n".join(all_tables)

def save_signature_tables(csv_file_path, filename_prefix='signature_performance', aggregation='mean'):
    """Save all 4 signature tables to separate files."""
    
    tables = {
        'keygen_time_ms': 'keygen',
        'sign_time_ms': 'sign',
        'verify_time_ms': 'verify',
        'overhead_bytes': 'overhead'
    }
    
    saved_files = []
    
    for column_name, file_suffix in tables.items():
        print(f"Generating signature {column_name} table...")
        table_content = generate_performance_table(csv_file_path, column_name, aggregation, 'signature')
        
        filename = f"{filename_prefix}_{file_suffix}.md"
        with open(filename, 'w') as f:
            f.write(table_content)
        
        saved_files.append(filename)
        print(f"Saved: {filename}")
    
    return saved_files

def save_kem_tables(csv_file_path, filename_prefix='kem_performance', aggregation='mean'):
    """Save all 4 KEM tables to separate files."""
    
    tables = {
        'keygen_time_ms': 'keygen',
        'encap_time_ms': 'encap',
        'decap_time_ms': 'decap',
        'overhead_bytes': 'overhead'
    }
    
    saved_files = []
    
    for column_name, file_suffix in tables.items():
        print(f"Generating KEM {column_name} table...")
        table_content = generate_performance_table(csv_file_path, column_name, aggregation, 'kem')
        
        filename = f"{filename_prefix}_{file_suffix}.md"
        with open(filename, 'w') as f:
            f.write(table_content)
        
        saved_files.append(filename)
        print(f"Saved: {filename}")
    
    return saved_files

def analyze_csv_structure(csv_file_path):
    """Analyze CSV structure to see what algorithms and columns are available."""
    data = pd.read_csv(csv_file_path)
    
    print("CSV Analysis:")
    print("="*50)
    print(f"Total rows: {len(data)}")
    print(f"Columns: {list(data.columns)}")
    
    if 'type' in data.columns:
        print(f"\nTypes found: {data['type'].unique()}")
    
    if 'algorithm' in data.columns:
        print(f"\nNumber of unique algorithms: {data['algorithm'].nunique()}")
        print("Algorithms found:")
        for alg in sorted(data['algorithm'].unique()):
            print(f"  - {alg}")
    
    if 'text_size_kb' in data.columns:
        print(f"\nText sizes: {sorted(data['text_size_kb'].unique())}")
    
    if 'security_level_bits' in data.columns:
        print(f"Security levels: {sorted(data['security_level_bits'].unique())}")
    
    return data

# Usage examples:
"""
# Analyze your CSV first
data_info = analyze_csv_structure('your_benchmark_results.csv')

# For Digital Signatures:
# Generate individual signature table
keygen_sig_table = generate_performance_table('signature_benchmark_results.csv', 'keygen_time_ms', 'mean', 'signature')
print(keygen_sig_table)

# Generate all signature tables
all_sig_tables = generate_signature_tables('signature_benchmark_results.csv', 'mean')
print(all_sig_tables)

# Save all signature tables
save_signature_tables('signature_benchmark_results.csv', 'sig_results', 'mean')

# For KEM:
# Generate individual KEM table
keygen_kem_table = generate_performance_table('kem_benchmark_results.csv', 'keygen_time_ms', 'mean', 'kem')
print(keygen_kem_table)

# Generate all KEM tables
all_kem_tables = generate_kem_tables('kem_benchmark_results.csv', 'mean')
print(all_kem_tables)

# Save all KEM tables
save_kem_tables('kem_benchmark_results.csv', 'kem_results', 'mean')

# If you have both in one file:
# Check what's available first
analyze_csv_structure('combined_benchmark_results.csv')

# Then generate tables for both types
sig_tables = generate_signature_tables('combined_benchmark_results.csv', 'mean')
kem_tables = generate_kem_tables('combined_benchmark_results.csv', 'mean')
"""

save_kem_tables('pq_kem_benchmark_20250909_140634.csv', 'kem_results', 'mean')


Generating KEM keygen_time_ms table...
Saved: kem_results_keygen.md
Generating KEM encap_time_ms table...
Saved: kem_results_encap.md
Generating KEM decap_time_ms table...
Saved: kem_results_decap.md
Generating KEM overhead_bytes table...
Saved: kem_results_overhead.md


['kem_results_keygen.md',
 'kem_results_encap.md',
 'kem_results_decap.md',
 'kem_results_overhead.md']